# FT1 — fine-tune Qwen3.5-9B on H3b traces (Colab G4 / RTX PRO 6000)

**Before running**
1. *Runtime → Change runtime type* → **RTX PRO 6000 (G4)** (High-RAM if offered). A100 also works (use `RESUME` after disconnects).
2. 🔑 *Secrets* (left sidebar) → add **`GH_TOKEN`** = a GitHub fine-grained token with *Contents: read* on `JimmyK300/zalo-ft1-data`; enable notebook access.
3. Run cells top to bottom. **Run the smoke test (step 5) before the full run (step 6).**

Locked method: LoRA r=16/alpha=32 on all linear layers, assistant-only loss, no reasoning, 2 epochs, lr 1e-4 cosine.
Outputs go to Drive so a disconnect never loses checkpoints.

## 0. Settings

In [ ]:
DATA_OPTION = 'opt2'      # 'opt2' = train + final-test businesses (recommended), 'opt1' = train businesses only
FOUR_BIT    = False       # True only on a 24 GB GPU (L4)
RESUME      = True       # True after a disconnect: continues the full run from its latest checkpoint
DRIVE_DIR   = '/content/drive/MyDrive/ZaloAI_VECNA/ft1'   # shared folder (add a shortcut to My Drive if it isn't yours)

## 1. GPU check, Drive, token

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.makedirs(DRIVE_DIR, exist_ok=True)  # ft1/ is created here; training data comes from GitHub, not Drive
print('Drive output folder:', DRIVE_DIR)

## 2. Code (public branch) and training data (private repo)

In [ ]:
import subprocess
!rm -rf /content/zalo-vecna /content/ft1data
!git clone -q -b harness-h3b https://github.com/JimmyK300/zalo-vecna.git /content/zalo-vecna
!git clone -q https://$GH_TOKEN@github.com/JimmyK300/zalo-ft1-data.git /content/ft1data
!gunzip -c /content/ft1data/sft_nothink_{DATA_OPTION}.jsonl.gz > /content/sft_nothink.jsonl
n = sum(1 for _ in open('/content/sft_nothink.jsonl'))
print('examples:', n, '(expected 2634 for opt2, 1928 for opt1)')
!cd /content/zalo-vecna && git log --oneline -1

## 3. Install (5–15 min; `causal-conv1d` compiles)
The kernels make Qwen3.5's 24 linear-attention layers fast. If they fail to install, training still runs, only slower.

In [ ]:
!pip -q install -U "transformers>=5.2" peft accelerate safetensors
!pip -q uninstall -y torchao   # Colab ships torchao 0.10; new peft refuses LoRA when torchao < 0.16 is installed (we do not use it)
!pip -q install flash-linear-attention causal-conv1d || echo "WARNING: kernel install failed - training will be slower"
import importlib.util
for m in ['transformers', 'peft', 'fla', 'causal_conv1d']:
    print(m, 'OK' if importlib.util.find_spec(m) else 'MISSING')

In [ ]:
!pip uninstall -y torchao

## 4. Common command

In [ ]:
import shlex
def train_cmd(out, extra=''):
    flags = ' --four-bit' if FOUR_BIT else ''
    return (f"cd /content/zalo-vecna && python finetune/train_sft.py --data /content/sft_nothink.jsonl "
            f"--out {shlex.quote(out)}{flags} {extra}")
print(train_cmd(DRIVE_DIR + '/run'))

## 5. Smoke test (~10 min, 50 examples) — **do not skip**
Check in the output:
- `trainable params: ...` (model + LoRA loaded)
- **`first optimizer step after Ns`**: healthy is roughly 1–3 min. Over 10 min means the linear-attention kernels did not load.
- no `CUDA out of memory`, ends with `done`.

The first run also downloads the model (~18 GB).

In [ ]:
cmd = train_cmd(DRIVE_DIR + '/smoke', '--limit 50 --epochs 1 --sketch-examples 8')
!{cmd} 2>&1 | tee /content/smoke.log
!grep -E "examples,|trainable|first optimizer|out of memory|done" /content/smoke.log

## 6. Full run (~5 h on G4)
Keep the tab open. Progress lines appear about every 10%; checkpoints are saved to Drive.
**If Colab disconnects:** reconnect, set `RESUME = True` in step 0, run steps 0–4, then this cell again.

In [ ]:
!CAUSAL_CONV1D_FORCE_BUILD=TRUE pip install --no-cache-dir --no-binary causal-conv1d --force-reinstall --no-deps --no-build-isolation causal-conv1d
!python -c "import causal_conv1d; print('causal_conv1d OK')"

In [ ]:
!pip install causal-conv1d --no-build-isolation 2>&1 | tail -3
!python -c "import causal_conv1d; print('causal_conv1d OK')"
cmd = train_cmd(DRIVE_DIR + '/run', '--resume --sketch-examples 256')
!{cmd} 2>&1 | tee -a {DRIVE_DIR}/run.log

## 7. Progress / health (safe to run any time, e.g. from another tab after reconnecting)

In [ ]:
import json, os
log = f'{DRIVE_DIR}/run/train_log.jsonl'
if os.path.exists(log):
    rows = [json.loads(l) for l in open(log)]
    cfg = json.load(open(f'{DRIVE_DIR}/run/run_config.json'))
    last = rows[-1]
    print(f"step {last['step']}/{cfg['total_steps']}  epoch {last['epoch']}  loss {last['loss']:.4f}  "
          f"elapsed {last['sec']/3600:.2f} h  ETA {(cfg['total_steps']-last['step'])*last['sec']/max(last['step'],1)/3600:.2f} h")
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(7, 3)); plt.plot([r['step'] for r in rows], [r['loss'] for r in rows]); plt.xlabel('step'); plt.ylabel('loss'); plt.show()
    except Exception as e:
        print(e)
print(sorted(p for p in os.listdir(f'{DRIVE_DIR}/run')) if os.path.exists(f'{DRIVE_DIR}/run') else 'no run yet')

## 8. Done
When step 6 prints `done`, Drive has `ft1/run/`: `final/` (adapter), `ckpt_step*/`, `train_log.jsonl`, `example_losses.jsonl`,
`grad_sketch_mid.npy`, `grad_sketch_final.npy`, `grad_index.json`, `run_config.json`.
Bring these back for evaluation and the weight-ledger entry.

(Optional) Evaluation in this notebook uses the `ZaloAI_VECNA/train` folder on Drive:
```
!cd /content/zalo-vecna && python finetune/eval_h3b_hf.py --data /content/drive/MyDrive/ZaloAI_VECNA/train --ids finetune/eval80.json --out base_eval.jsonl
!cd /content/zalo-vecna && python finetune/eval_h3b_hf.py --data /content/drive/MyDrive/ZaloAI_VECNA/train --ids finetune/eval80.json --adapter {DRIVE_DIR}/run/final --out ft1_eval.jsonl
```